# 🛠️ Notebook 2: Hands-on Prompting Lab
## Building Prompt Variants & Experimenting with Techniques

**What you'll do:**
- Build and iterate on prompts for real tasks
- Compare variant outputs side-by-side
- Experiment with temperature, system prompts, and formatting
- Create a prompt engineering toolkit

**Model:** `gpt-4o-mini` | **Embeddings:** `text-embedding-3-small`

---

In [ ]:
# ============================================
# 📦 Setup
# ============================================
!pip install openai tiktoken rich numpy -q

import os, json, time, textwrap
from getpass import getpass
from openai import OpenAI
import numpy as np
from IPython.display import display, HTML, Markdown as IPyMarkdown

os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API key: ")
client = OpenAI()

MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

def chat(messages, temperature=0.7, max_tokens=1024):
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=text)
    return r.data[0].embedding

def compare_prompts(prompts_dict, temperature=0.5):
    """Run multiple prompts and display results side by side."""
    results = {}
    for name, msgs in prompts_dict.items():
        if isinstance(msgs, str):
            msgs = [{"role": "user", "content": msgs}]
        results[name] = chat(msgs, temperature=temperature)
    
    for name, result in results.items():
        display(IPyMarkdown(f"### 🔹 {name}\n---\n{result}\n"))
    return results

print("✅ Ready! Let's build some prompts.")

---
## 🧪 Lab 1: The Prompt Anatomy Workshop

Every effective prompt has building blocks. Let's dissect and build them systematically.

In [ ]:
# ============================================
# 🔬 Lab 1.1: Deconstructing a Prompt
# ============================================

# We'll build the SAME task with increasing prompt sophistication
task = "Write a product description for a smart water bottle that tracks hydration."

# Level 1: Bare minimum
level1 = task

# Level 2: Add context + constraints
level2 = f"""You are a senior copywriter at a DTC e-commerce brand.

{task}

Requirements:
- Target audience: health-conscious millennials
- Tone: friendly but premium
- Length: 80-100 words
- Include a call-to-action"""

# Level 3: Add examples + structure
level3 = f"""You are a senior copywriter at a DTC e-commerce brand known for witty, 
benefit-driven copy.

Here's an example of our brand voice:
"Your morning coffee deserves better than a paper cup. Meet ThermoLux — 
12 hours hot, 24 hours cold, infinite style points. ☕"

Now {task.lower()}

Format:
**Headline:** [punchy, under 8 words]
**Body:** [80-100 words, benefit-focused, conversational]
**CTA:** [action-oriented button text]

Constraints:
- Target: health-conscious millennials (25-35)
- Avoid: jargon, clichés like "game-changer" or "revolutionary"
- Must mention: the app integration feature"""

# Level 4: Add persona + anti-patterns + scoring criteria
level4 = f"""<role>Senior copywriter at HydroTech, a premium DTC wellness brand. 
Your copy has won 3 Webby Awards. You write like Apple meets Glossier.</role>

<task>{task}</task>

<brand_voice_examples>
GOOD: "Your morning coffee deserves better than a paper cup. Meet ThermoLux — 12 hours hot, 24 hours cold, infinite style points."
GOOD: "Track every sip. Crush every goal. Look good doing it."
BAD: "This revolutionary game-changing bottle will transform your hydration journey."
BAD: "Buy now and experience the future of drinking water."
</brand_voice_examples>

<output_format>
**Headline:** [punchy, under 8 words, no clichés]
**Subheadline:** [benefit-driven, 10-15 words]
**Body:** [80-100 words]
**CTA Button:** [3-5 words, action-oriented]
**Meta Description:** [under 155 characters for SEO]
</output_format>

<constraints>
- Target: health-conscious millennials (25-35)
- MUST mention: app sync, daily goal tracking, premium materials
- AVOID: "revolutionary", "game-changer", "transform", "journey", exclamation marks
- Tone: confident, warm, slightly playful
</constraints>

<scoring_criteria>
Rate your own output 1-5 on: Clarity, Brand Fit, Persuasion, Originality
If any score < 4, revise before submitting.
</scoring_criteria>"""

results = compare_prompts({
    "Level 1 — Bare Minimum": level1,
    "Level 2 — Context + Constraints": level2,
    "Level 3 — Examples + Structure": level3,
    "Level 4 — Full Framework": level4
}, temperature=0.5)

---
## 🧪 Lab 2: Temperature & Creativity Experiments

In [ ]:
# ============================================
# 🌡️ Lab 2.1: Temperature Sweep
# ============================================
prompt = "Write a one-paragraph company bio for an AI startup called 'NeuralForge' that builds developer tools."

temps = [0.0, 0.3, 0.7, 1.0, 1.5]
temp_results = {}

for t in temps:
    response = chat(
        [{"role": "user", "content": prompt}],
        temperature=t
    )
    temp_results[t] = response
    display(IPyMarkdown(f"### 🌡️ Temperature = {t}\n{response}\n"))

# Measure diversity using embeddings
print("\n📐 Semantic Similarity Between Outputs:")
embs = {t: embed(r) for t, r in temp_results.items()}
temps_list = list(embs.keys())
for i in range(len(temps_list)):
    for j in range(i+1, len(temps_list)):
        sim = np.dot(embs[temps_list[i]], embs[temps_list[j]]) / (
            np.linalg.norm(embs[temps_list[i]]) * np.linalg.norm(embs[temps_list[j]]))
        print(f"  T={temps_list[i]:.1f} ↔ T={temps_list[j]:.1f}  similarity: {sim:.4f}")

In [ ]:
# ============================================
# 🎭 Lab 2.2: Persona Variations
# ============================================
topic = "Explain why code reviews are important."

personas = {
    "👨‍💼 Corporate Manager": "You are a Fortune 500 engineering VP writing a memo to your department.",
    "🏴‍☠️ Pirate": "You are a pirate captain explaining things in pirate speak. Arr!",
    "👶 For a 10-year-old": "You are a friendly teacher explaining to a 10-year-old. Use simple words and analogies.",
    "📜 Shakespeare": "You are Shakespeare. Respond in iambic pentameter with dramatic flair.",
    "🤖 Sarcastic AI": "You are a sarcastic AI who is technically helpful but can't help being witty about it."
}

for name, system_prompt in personas.items():
    response = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": topic}
    ], temperature=0.7, max_tokens=200)
    display(IPyMarkdown(f"### {name}\n{response}\n"))

---
## 🧪 Lab 3: Prompt Patterns Cookbook

In [ ]:
# ============================================
# 📖 Lab 3.1: The "Flipped Interaction" Pattern
# ============================================
# Instead of answering, make the AI ASK the right questions

flipped_prompt = """I want to write a great README for my open-source project. 
Instead of writing it for me, ask me the 7 most important questions you need 
answered to write an excellent README. Ask them one at a time — start with the first."""

response = chat([{"role": "user", "content": flipped_prompt}], temperature=0.5)
display(IPyMarkdown(f"### Flipped Interaction Pattern\n{response}"))

In [ ]:
# ============================================
# 🧩 Lab 3.2: The "Constraints Breed Creativity" Pattern
# ============================================
base_task = "Write a tagline for a coffee shop."

constraints_levels = {
    "No constraints": base_task,
    "3 words max": f"{base_task} It must be exactly 3 words.",
    "Alliterative": f"{base_task} Every word must start with the same letter.",
    "No letter 'e'": f"{base_task} You cannot use the letter 'e' anywhere.",
    "Haiku format": f"{base_task} Format it as a haiku (5-7-5 syllables).",
}

for name, prompt in constraints_levels.items():
    response = chat([{"role": "user", "content": prompt}], temperature=0.8, max_tokens=100)
    display(IPyMarkdown(f"**{name}:** {response.strip()}"))

In [ ]:
# ============================================
# 🔄 Lab 3.3: Iterative Refinement Pattern
# ============================================
messages = [
    {"role": "system", "content": "You are an expert email copywriter. After each draft, rate it 1-10 on: clarity, persuasion, conciseness. Then suggest specific improvements."},
    {"role": "user", "content": "Write a cold outreach email to a VP of Engineering about our AI code review tool. Keep it under 100 words."}
]

for iteration in range(3):
    response = chat(messages, temperature=0.5)
    messages.append({"role": "assistant", "content": response})
    display(IPyMarkdown(f"### Iteration {iteration + 1}\n{response}"))
    
    if iteration < 2:
        refine_msg = "Good. Now rewrite addressing your own improvement suggestions. Be more concise and compelling."
        messages.append({"role": "user", "content": refine_msg})
        print(f"--- Refining... ---\n")

In [ ]:
# ============================================
# 🏗️ Lab 3.4: Build Your Own Prompt Template
# ============================================
def prompt_builder(task, context="", examples=None, output_format="", constraints=None, role=""):
    """Assemble a structured prompt from components."""
    sections = []
    
    if role:
        sections.append(f"<role>{role}</role>")
    if context:
        sections.append(f"<context>{context}</context>")
    
    sections.append(f"<task>{task}</task>")
    
    if examples:
        ex_str = "\n".join([f"Input: {e['in']}\nOutput: {e['out']}" for e in examples])
        sections.append(f"<examples>\n{ex_str}\n</examples>")
    
    if output_format:
        sections.append(f"<output_format>{output_format}</output_format>")
    
    if constraints:
        c_str = "\n".join([f"- {c}" for c in constraints])
        sections.append(f"<constraints>\n{c_str}\n</constraints>")
    
    return "\n\n".join(sections)

# Demo usage
my_prompt = prompt_builder(
    task="Summarize this article into 3 bullet points",
    role="Senior tech journalist at Wired",
    context="The article discusses OpenAI's latest model release and its implications for enterprise AI adoption.",
    output_format="3 bullet points, each 15-20 words. Start each with an action verb.",
    constraints=[
        "No jargon — a business executive should understand every word",
        "Include at least one specific number or statistic",
        "Final bullet must address future implications"
    ]
)

print("📋 Generated Prompt:\n")
print(my_prompt)
print("\n" + "="*60 + "\n")

response = chat([{"role": "user", "content": my_prompt}], temperature=0.3)
display(IPyMarkdown(f"### Output\n{response}"))

---
## 🧪 Lab 4: Embedding-Powered Prompt Selection

In [ ]:
# ============================================
# 🎯 Lab 4.1: Automatic Prompt Routing
# ============================================
# Use embeddings to match user queries to the best prompt template

prompt_templates = {
    "code_review": {
        "description": "Review code for bugs, style issues, and improvements",
        "template": "You are a senior software engineer. Review this code for bugs, performance issues, and style. Provide specific line-by-line feedback.\n\nCode:\n{input}"
    },
    "creative_writing": {
        "description": "Write creative content like stories, poems, marketing copy",
        "template": "You are an award-winning creative writer. Write engaging, vivid content that captivates the reader.\n\nTask: {input}"
    },
    "data_analysis": {
        "description": "Analyze data, statistics, trends, and create insights",
        "template": "You are a data analyst. Analyze the following data/question with precision. Include specific numbers and actionable insights.\n\nAnalysis request: {input}"
    },
    "explanation": {
        "description": "Explain complex topics simply and clearly",
        "template": "You are an expert teacher known for clear explanations. Break this down step by step using analogies.\n\nExplain: {input}"
    },
}

# Pre-compute template embeddings
template_embeddings = {
    name: embed(info["description"]) 
    for name, info in prompt_templates.items()
}

def route_prompt(user_query):
    """Find the best prompt template for a given query."""
    query_emb = embed(user_query)
    scores = {}
    for name, templ_emb in template_embeddings.items():
        scores[name] = np.dot(query_emb, templ_emb) / (
            np.linalg.norm(query_emb) * np.linalg.norm(templ_emb))
    best = max(scores, key=scores.get)
    return best, scores

# Test routing
test_queries = [
    "Can you check my Python function for any issues?",
    "Write me a catchy tagline for a dog walking app",
    "What trends do you see in this quarterly sales data?",
    "How does quantum computing actually work?"
]

print("🎯 Prompt Routing Results\n")
for q in test_queries:
    best, scores = route_prompt(q)
    print(f'Query: "{q}"')
    print(f'  → Routed to: **{best}** (score: {scores[best]:.4f})')
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        bar = "█" * int(score * 40)
        print(f"    {name:>20s}: {score:.4f} {bar}")
    print()

---
## ✅ Lab Complete!

**Key Takeaways:**
1. **Prompt anatomy matters** — Role, context, examples, format, and constraints each improve output quality
2. **Temperature controls creativity** — Use 0 for factual tasks, 0.7-1.0 for creative ones
3. **Iteration beats perfection** — Start simple, refine based on output
4. **Embeddings enable smart routing** — Match queries to optimal prompt templates automatically

> 📌 **Next:** Move to **Notebook 3** for evaluation and optimization!